# Maintainer: Regenerate the Northgate Advisory Sample Data

**Workshop facilitators don't need this notebook.** The sample data is shipped, ready-made, in the `data/` folder next to it, and the [1. Workshop Data Setup](#notebook-3120290690157533) notebook copies it into the workshop volume.

Run this notebook only to change the scenario (people, clients, engagements, anomalies). It writes everything to `data/`:

| Folder | Contents |
| --- | --- |
| `tables/` | The four master data tables (`employees`, `clients`, `engagements`, `engagement_assignments`) as JSON lines |
| `timesheets/`, `expenses/`, `reference/` | The raw CSV and JSON files for August-October 2026 |
| `contracts/` | 16 Statement of Work PDFs and 1 change request |
| `_facilitator/` | The answer key for the AI extraction exercise |
| `new_month/` | The November timesheet file for the scheduling exercise |
| `designer_demo/` | The pre-built Designer prep for the PDF extension demo. It's built by hand, so this notebook only copies the saved version |
| `MANIFEST.json` | A checksum for every file and the table schemas |

The data uses a fixed random seed and each cell restarts from a saved checkpoint, so every run produces byte-identical files. If you change the scenario, run all cells, run [1. Workshop Data Setup](#notebook-3120290690157533) again, then update the answer keys in the tutorial notebooks (for example 4,152 October billable hours and GBP 533,519 WIP).

## Generate the dataset

In [0]:
%pip install fpdf2 --quiet

In [0]:
import os

# Everything is written to the data/ folder next to this notebook. The setup notebook copies that folder into the workshop volume.
DATA_DIR = os.path.join(os.getcwd(), "data")
VOLUME_PATH = DATA_DIR  # the generator cells below write to f"{VOLUME_PATH}/<folder>/..."

for sub in ["timesheets", "expenses", "reference", "contracts", "_facilitator", "new_month", "tables"]:
    os.makedirs(f"{DATA_DIR}/{sub}", exist_ok=True)

print(f"Writing the sample data to {DATA_DIR}")

In [0]:
import random
from datetime import date, timedelta

# A private generator, so Spark's own internal use of the `random` module can't change the generated data
rng = random.Random(2026)

FIRM_NAME = "Northgate Advisory LLP"
HOURS_PER_DAY = 7.5
PERIOD_START, PERIOD_END = date(2026, 8, 1), date(2026, 10, 31)
UK_BANK_HOLIDAYS = {date(2026, 8, 31)}

PRACTICES = ["Audit", "Technology Consulting", "Deal Advisory", "Risk Consulting", "Tax"]
OFFICES = ["London", "Manchester", "Birmingham", "Leeds", "Edinburgh"]

# grade, level, standard day rate, cost day rate, utilisation target, headcount
GRADES = [
    ("Partner",           8, 2400, 1100, 0.40, 5),
    ("Director",          7, 1950,  850, 0.55, 5),
    ("Senior Manager",    6, 1600,  650, 0.65, 8),
    ("Manager",           5, 1300,  520, 0.75, 10),
    ("Assistant Manager", 4, 1050,  400, 0.80, 8),
    ("Senior Consultant", 3,  850,  320, 0.85, 10),
    ("Consultant",        2,  650,  250, 0.85, 9),
    ("Analyst",           1,  480,  190, 0.85, 5),
]
GRADE_INFO = {g[0]: dict(level=g[1], standard_rate=g[2], cost_rate=g[3], utilisation_target=g[4]) for g in GRADES}

def working_days(start, end):
    d = start
    while d <= end:
        if d.weekday() < 5 and d not in UK_BANK_HOLIDAYS:
            yield d
        d += timedelta(days=1)

def round_half(x):
    return max(0.0, round(x * 2) / 2)

# ---------------- Employees ----------------
FIRST_NAMES = ["Olivia", "James", "Amelia", "Oliver", "Isla", "Harry", "Ava", "George", "Mia", "Noah",
               "Sophia", "Jack", "Grace", "Leo", "Freya", "Arthur", "Emily", "Oscar", "Ella", "Charlie",
               "Priya", "Arjun", "Aisha", "Mohammed", "Chloe", "Daniel", "Hannah", "Ethan", "Zara", "Samuel",
               "Niamh", "Callum", "Fatima", "Lucas", "Ruby", "Thomas", "Evie", "Rohan", "Maya", "Finlay"]
LAST_NAMES = ["Smith", "Jones", "Taylor", "Brown", "Williams", "Wilson", "Johnson", "Davies", "Patel", "Robinson",
              "Wright", "Thompson", "Evans", "Walker", "White", "Roberts", "Green", "Hall", "Khan", "Clarke",
              "Lewis", "Harris", "Martin", "Jackson", "Wood", "Turner", "Hughes", "Shah", "Murphy", "Campbell",
              "Stewart", "Kaur", "Ahmed", "Edwards", "Collins", "Bennett", "Mitchell", "Reid", "Fraser", "Singh"]
name_pool = rng.sample([(f, l) for f in FIRST_NAMES for l in LAST_NAMES], 60)

employees = []
for grade, level, std, cost, util, headcount in GRADES:
    for i in range(headcount):
        first, last = name_pool[len(employees)]
        tenure_years = max(0.3, level * 1.6 + rng.uniform(-1.5, 2.5))
        employees.append(dict(
            employee_id=f"EMP{1001 + len(employees)}",
            full_name=f"{first} {last}", first_name=first, last_name=last,
            email=f"{first.lower()}.{last.lower()}@northgate-advisory.example",
            grade=grade, grade_level=level, practice=PRACTICES[i % len(PRACTICES)],
            office=rng.choice(OFFICES), manager_id=None,
            hire_date=PERIOD_START - timedelta(days=int(tenure_years * 365)),
            employment_type="Permanent", annual_utilisation_target=util, is_active=True))

# Line manager = someone in the same practice at the nearest higher grade
for e in employees:
    if e["grade_level"] == 8:
        continue
    seniors = [m for m in employees if m["practice"] == e["practice"] and m["grade_level"] > e["grade_level"]]
    nearest = min(m["grade_level"] for m in seniors)
    e["manager_id"] = rng.choice([m for m in seniors if m["grade_level"] == nearest])["employee_id"]

for e in rng.sample([e for e in employees if e["grade_level"] in (2, 3)], 3):
    e["employment_type"] = "Contractor"

emp_by_id = {e["employee_id"]: e for e in employees}
partners = [e for e in employees if e["grade"] == "Partner"]

# ---------------- Clients ----------------
CLIENT_SPECS = [
    ("CL001", "Albion Retail Group plc",      "Retail & Consumer",            "London",     "United Kingdom", "GBP", "Strategic"),
    ("CL002", "Harbourline Bank plc",         "Financial Services",           "London",     "United Kingdom", "GBP", "Strategic"),
    ("CL003", "Meridian Energy Ltd",          "Energy & Utilities",           "Aberdeen",   "United Kingdom", "GBP", "Key"),
    ("CL004", "Calder Valley Water",          "Energy & Utilities",           "Leeds",      "United Kingdom", "GBP", "Key"),
    ("CL005", "Pennine Logistics Ltd",        "Transport & Logistics",        "Manchester", "United Kingdom", "GBP", "Standard"),
    ("CL006", "Thistle Insurance DAC",        "Insurance",                    "Dublin",     "Ireland",        "EUR", "Key"),
    ("CL007", "Sterling Life Sciences plc",   "Life Sciences",                "Cambridge",  "United Kingdom", "GBP", "Strategic"),
    ("CL008", "Crown Borough Council",        "Public Sector",                "Birmingham", "United Kingdom", "GBP", "Standard"),
    ("CL009", "Northstar Telecom Inc.",       "Technology, Media & Telecoms", "New York",   "United States",  "USD", "Key"),
    ("CL010", "Granite Construction Group",   "Industrials",                  "Glasgow",    "United Kingdom", "GBP", "Standard"),
    ("CL011", "Bluewater Hotels Ltd",         "Leisure & Hospitality",        "Edinburgh",  "United Kingdom", "GBP", "Standard"),
    ("CL012", "Oakridge Asset Management",    "Financial Services",           "London",     "United Kingdom", "GBP", "Key"),
]
clients = [dict(client_id=c[0], client_name=c[1], industry=c[2], city=c[3], country=c[4], billing_currency=c[5],
                client_tier=c[6], relationship_partner_id=rng.choice(partners)["employee_id"],
                client_since=date(rng.randint(2012, 2024), rng.randint(1, 12), 1)) for c in CLIENT_SPECS]
client_by_id = {c["client_id"]: c for c in clients}

# ---------------- Engagements ----------------
ENGAGEMENT_SPECS = [
    ("CL001", "Audit",                 "FY26 Statutory Audit",                    "Fixed Fee",        date(2026, 7, 1),  date(2026, 11, 30)),
    ("CL002", "Risk Consulting",       "Internal Audit Co-source 2026",           "Time & Materials", date(2026, 4, 1),  date(2027, 3, 31)),
    ("CL002", "Technology Consulting", "Cloud Data Platform Strategy",            "Capped T&M",       date(2026, 8, 3),  date(2026, 10, 30)),
    ("CL003", "Deal Advisory",         "Financial Due Diligence - Project Falcon", "Time & Materials", date(2026, 8, 10), date(2026, 9, 25)),
    ("CL004", "Risk Consulting",       "Operational Resilience Review",           "Capped T&M",       date(2026, 7, 6),  date(2026, 10, 16)),
    ("CL005", "Technology Consulting", "ERP Implementation Assurance",            "Time & Materials", date(2026, 6, 1),  date(2026, 12, 18)),
    ("CL006", "Audit",                 "FY26 Interim Audit Procedures",           "Fixed Fee",        date(2026, 9, 1),  date(2026, 10, 30)),
    ("CL007", "Tax",                   "R&D Tax Credit Claim FY25",               "Fixed Fee",        date(2026, 8, 3),  date(2026, 9, 30)),
    ("CL007", "Deal Advisory",         "Carve-out Support - Project Heron",       "Time & Materials", date(2026, 9, 7),  date(2026, 12, 11)),
    ("CL008", "Risk Consulting",       "Procurement Fraud Risk Assessment",       "Capped T&M",       date(2026, 8, 17), date(2026, 10, 9)),
    ("CL009", "Technology Consulting", "Cyber Maturity Assessment",               "Capped T&M",       date(2026, 9, 14), date(2026, 11, 27)),
    ("CL010", "Tax",                   "Transfer Pricing Documentation",          "Time & Materials", date(2026, 7, 13), date(2026, 10, 23)),
    ("CL011", "Audit",                 "FY26 Statutory Audit",                    "Fixed Fee",        date(2026, 8, 24), date(2026, 11, 13)),
    ("CL012", "Deal Advisory",         "Vendor Due Diligence - Project Kestrel",  "Time & Materials", date(2026, 8, 3),  date(2026, 10, 2)),
    ("CL012", "Tax",                   "VAT Compliance Review",                   "Time & Materials", date(2026, 9, 21), date(2026, 12, 4)),
    ("CL003", "Technology Consulting", "SAP S/4HANA Data Migration Assurance",    "Capped T&M",       date(2026, 10, 5), date(2027, 2, 26)),
]
DISCOUNTS = {"Strategic": [0.10, 0.15, 0.20], "Key": [0.05, 0.10, 0.12], "Standard": [0.0, 0.05]}
EXPENSE_POLICIES = ["Expenses recharged at cost", "Expenses recharged at cost, capped at 8% of fees", "Expenses included in fees"]
OVERRUN_CODES = {"ENG-26-003", "ENG-26-007", "ENG-26-010"}  # deliberately under-priced
TODAY = date(2026, 11, 6)

leads_by_practice = {p: sorted([e for e in employees if e["practice"] == p and e["grade_level"] >= 7], key=lambda e: -e["grade_level"]) for p in PRACTICES}
managers_by_practice = {p: [e for e in employees if e["practice"] == p and e["grade_level"] in (5, 6)] for p in PRACTICES}

engagements, sow_terms, practice_count = [], {}, {}
for i, (client_id, practice, name, billing_type, start, end) in enumerate(ENGAGEMENT_SPECS, start=1):
    code = f"ENG-26-{i:03d}"
    n = practice_count.get(practice, 0)
    practice_count[practice] = n + 1
    lead = leads_by_practice[practice][n % len(leads_by_practice[practice])]
    mgr = managers_by_practice[practice][n % len(managers_by_practice[practice])]
    engagements.append(dict(
        engagement_code=code, engagement_name=name, engagement_type="Client", client_id=client_id,
        practice=practice, billing_type=billing_type,
        engagement_partner_id=lead["employee_id"], engagement_manager_id=mgr["employee_id"],
        start_date=start, end_date=end, status="Completed" if end < TODAY else "Active",
        sow_reference=f"SOW-26-{i:03d}"))
    # Commercial terms live ONLY in the SOW PDFs, not in the engagements table
    sow_terms[code] = dict(
        discount=rng.choice(DISCOUNTS[client_by_id[client_id]["client_tier"]]),
        payment_terms_days=rng.choice([30, 45, 60]),
        purchase_order=f"PO-{rng.randint(400000, 499999)}",
        expense_policy=rng.choice(EXPENSE_POLICIES))

for code, name in [("INT-BD", "Business Development"), ("INT-TRN", "Training & CPD"),
                   ("INT-ADM", "Internal Administration"), ("INT-LVE", "Annual Leave")]:
    engagements.append(dict(
        engagement_code=code, engagement_name=name, engagement_type="Internal", client_id=None,
        practice="Firmwide", billing_type="Non-chargeable", engagement_partner_id=None,
        engagement_manager_id=None, start_date=date(2026, 1, 1), end_date=date(2026, 12, 31),
        status="Active", sow_reference=None))
eng_by_code = {e["engagement_code"]: e for e in engagements}
client_engagements = [e for e in engagements if e["engagement_type"] == "Client"]

# ---------------- Assignments (staffing) ----------------
assignments, load = [], {e["employee_id"]: [] for e in employees}

def overlap_load(emp_id, start, end):
    return sum(a for (s, t, a) in load[emp_id] if s <= end and t >= start)

def add_assignment(eng, emp, role, alloc):
    assignments.append(dict(
        assignment_id=f"ASG-{len(assignments) + 1:04d}", engagement_code=eng["engagement_code"],
        employee_id=emp["employee_id"], role=role, allocation_pct=alloc,
        assigned_from=eng["start_date"], assigned_to=eng["end_date"]))
    load[emp["employee_id"]].append((eng["start_date"], eng["end_date"], alloc))

for eng in client_engagements:
    add_assignment(eng, emp_by_id[eng["engagement_partner_id"]], "Engagement Partner", 0.1)
    add_assignment(eng, emp_by_id[eng["engagement_manager_id"]], "Engagement Manager", 0.4)
    # Prefer staff from the engagement's own practice; borrow from other practices if they're fully booked
    by_load = lambda e: (overlap_load(e["employee_id"], eng["start_date"], eng["end_date"]), rng.random())
    pool = sorted([e for e in employees if e["practice"] == eng["practice"] and e["grade_level"] <= 4], key=by_load) + \
           sorted([e for e in employees if e["practice"] != eng["practice"] and e["grade_level"] <= 4], key=by_load)
    team_size, added = rng.choice([3, 4]), 0
    for emp in pool:
        if added == team_size:
            break
        free = round(1.0 - overlap_load(emp["employee_id"], eng["start_date"], eng["end_date"]), 1)
        if free < 0.2:
            continue
        alloc = min(rng.choice([0.5, 0.6, 0.8, 1.0]), free)
        add_assignment(eng, emp, "In-charge" if emp["grade_level"] == 4 else "Team Member", alloc)
        added += 1

assignments_by_emp = {}
for a in assignments:
    assignments_by_emp.setdefault(a["employee_id"], []).append(a)

# ---------------- SOW pricing: agreed rates and fee ----------------
for eng in client_engagements:
    code, t = eng["engagement_code"], sow_terms[eng["engagement_code"]]
    team = [a for a in assignments if a["engagement_code"] == code]
    grades_on_team = sorted({emp_by_id[a["employee_id"]]["grade"] for a in team}, key=lambda g: -GRADE_INFO[g]["level"])
    t["day_rates"] = [dict(grade=g, standard_day_rate=GRADE_INFO[g]["standard_rate"], discount_pct=round(t["discount"] * 100),
                           agreed_day_rate=int(round(GRADE_INFO[g]["standard_rate"] * (1 - t["discount"]) / 5) * 5))
                      for g in grades_on_team]
    agreed = {r["grade"]: r["agreed_day_rate"] for r in t["day_rates"]}
    n_days = sum(1 for _ in working_days(eng["start_date"], eng["end_date"]))
    planned_fee = sum(a["allocation_pct"] * n_days * agreed[emp_by_id[a["employee_id"]]["grade"]] for a in team)
    factor = 0.85 if code in OVERRUN_CODES else rng.uniform(1.05, 1.25)
    t["fee_gbp"] = int(round(planned_fee * factor, -3))

# ---------------- Write the master data as JSON lines ----------------
# The setup notebook loads each file into a Delta table with this schema and comment
import json
TABLE_SPECS = {}

def save_table(rows, ddl, name, comment):
    cols = [c.strip().split()[0] for c in ddl.split(",")]
    with open(f"{DATA_DIR}/tables/{name}.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps({c: (r[c].isoformat() if isinstance(r[c], date) else r[c]) for c in cols}) + "\n")
    TABLE_SPECS[name] = dict(ddl=ddl, comment=comment, rows=len(rows))
    print(f"{name:<24} {len(rows):>4} rows")

save_table(employees,
    "employee_id STRING, full_name STRING, first_name STRING, last_name STRING, email STRING, grade STRING, grade_level INT, "
    "practice STRING, office STRING, manager_id STRING, hire_date DATE, employment_type STRING, annual_utilisation_target DOUBLE, is_active BOOLEAN",
    "employees", "HR master data. One row per employee. manager_id is the line manager (self-join to employee_id). Utilisation target is billable hours / available hours.")
save_table(clients,
    "client_id STRING, client_name STRING, industry STRING, city STRING, country STRING, billing_currency STRING, client_tier STRING, relationship_partner_id STRING, client_since DATE",
    "clients", "CRM client master. relationship_partner_id joins to employees.employee_id.")
save_table(engagements,
    "engagement_code STRING, engagement_name STRING, engagement_type STRING, client_id STRING, practice STRING, billing_type STRING, "
    "engagement_partner_id STRING, engagement_manager_id STRING, start_date DATE, end_date DATE, status STRING, sow_reference STRING",
    "engagements", "Engagement (project) master: client engagements plus internal non-chargeable time codes (INT-*). Fees and agreed day rates are only in the Statement of Work PDFs.")
save_table(assignments,
    "assignment_id STRING, engagement_code STRING, employee_id STRING, role STRING, allocation_pct DOUBLE, assigned_from DATE, assigned_to DATE",
    "engagement_assignments", "Resourcing: which employees are staffed on which engagement, their role and planned allocation (0-1).")

# Checkpoint the random generator, so each later cell can be re-run on its own and still produce exactly the same data
RNG_STATE = {"after_master_data": rng.getstate()}

In [0]:
import csv, glob, json, os

# Start from the checkpoint saved by the cell above, so re-running this cell gives the same data every time
rng.setstate(RNG_STATE["after_master_data"])

ACTIVITIES = {
    "Audit": ["Planning", "Fieldwork", "Controls Testing", "Substantive Testing", "Review", "Reporting"],
    "Risk Consulting": ["Scoping", "Interviews", "Controls Testing", "Analysis", "Reporting"],
    "Technology Consulting": ["Workshops", "Requirements", "Solution Design", "Testing", "PMO", "Reporting"],
    "Deal Advisory": ["Data Room Review", "Analysis", "Management Meetings", "Report Drafting", "Review"],
    "Tax": ["Data Gathering", "Computation", "Technical Research", "Review", "Client Meeting"],
}
COMMENTS = ["", "", "", "", "Client workshop", "Draft deliverable", "Team catch-up", "Review points cleared", "Travel to client site"]
TS_HEADER = ["timesheet_id", "employee_id", "engagement_code", "work_date", "hours", "activity", "billable", "approval_status", "comments"]
_ts_counter = [0]

def ts_row(emp, code, d, hours, billable, status, comment=None):
    _ts_counter[0] += 1
    eng = eng_by_code[code]
    activity = rng.choice(ACTIVITIES[eng["practice"]]) if eng["engagement_type"] == "Client" else eng["engagement_name"]
    return dict(timesheet_id=f"TS{_ts_counter[0]:07d}", employee_id=emp["employee_id"], engagement_code=code,
                work_date=d, hours=hours, activity=activity, billable=billable, approval_status=status,
                comments=rng.choice(COMMENTS) if comment is None else comment)

def approval_status_for(d, period_end):
    if (period_end - d).days <= 6:  # the latest week is still waiting for approval
        return "Submitted" if rng.random() < 0.4 else "Approved"
    r = rng.random()
    return "Approved" if r < 0.97 else ("Submitted" if r < 0.99 else "Rejected")

def generate_timesheet_rows(start, end):
    rows = []
    for emp in employees:
        for d in working_days(start, end):
            status = approval_status_for(d, end)
            if rng.random() < 0.04:
                rows.append(ts_row(emp, "INT-LVE", d, HOURS_PER_DAY, "N", status, "Annual leave"))
                continue
            active = [a for a in assignments_by_emp.get(emp["employee_id"], []) if a["assigned_from"] <= d <= a["assigned_to"]]
            day_rows = []
            for a in active:
                h = min(9.0, round_half(HOURS_PER_DAY * a["allocation_pct"] * rng.gauss(1.0, 0.18)))
                if h > 0:
                    write_off = rng.random() < 0.03
                    day_rows.append(ts_row(emp, a["engagement_code"], d, h, "N" if write_off else "Y", status,
                                           "Non-chargeable rework" if write_off else None))
            if day_rows and rng.random() < 0.08:  # occasional overtime
                rng.choice(day_rows)["hours"] += rng.choice([1.0, 1.5, 2.0, 3.0])
            remaining = HOURS_PER_DAY - sum(r["hours"] for r in day_rows)
            if remaining >= 0.5:
                if emp["grade_level"] >= 6:
                    code = "INT-BD" if rng.random() < 0.6 else "INT-ADM"
                else:
                    code = "INT-TRN" if rng.random() < 0.35 else "INT-ADM"
                day_rows.append(ts_row(emp, code, d, remaining, "N", status))
            rows.extend(day_rows)
    return rows

def messy_csv_values(r):
    """Simulate the formatting quirks of a timesheet-system export."""
    code, x = r["engagement_code"], rng.random()
    if x < 0.04:
        code = code + "  "
    elif x < 0.06:
        code = code.lower()
    billable = r["billable"].lower() if rng.random() < 0.06 else r["billable"]
    return [r["timesheet_id"], r["employee_id"], code, r["work_date"].strftime("%d/%m/%Y"), f"{r['hours']:.1f}",
            r["activity"], billable, r["approval_status"], r["comments"]]

def write_timesheet_files(rows, folder=None):
    folder = folder or f"{VOLUME_PATH}/timesheets"
    by_month = {}
    for r in sorted(rows, key=lambda r: (r["work_date"], r["employee_id"], r["timesheet_id"])):
        by_month.setdefault(r["work_date"].strftime("%Y_%m"), []).append(r)
    for month, month_rows in by_month.items():
        path = f"{folder}/timesheets_{month}.csv"
        with open(path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(TS_HEADER)
            w.writerows(messy_csv_values(r) for r in month_rows)
        print(f"{path}  ({len(month_rows)} rows)")

# ---------------- Timesheets with seeded anomalies ----------------
timesheet_rows = generate_timesheet_rows(PERIOD_START, PERIOD_END)
anomalies = {"over_12_hours": [], "weekend": [], "not_assigned": []}

for r in rng.sample([r for r in timesheet_rows if r["billable"] == "Y" and r["approval_status"] == "Approved"], 8):
    r["hours"] = rng.choice([13.0, 14.0, 15.5, 16.0])
    anomalies["over_12_hours"].append(r["timesheet_id"])

staff_assignments = [a for a in assignments if a["role"] != "Engagement Partner"]
weekend_days = [PERIOD_START + timedelta(days=i) for i in range((PERIOD_END - PERIOD_START).days + 1)
                if (PERIOD_START + timedelta(days=i)).weekday() >= 5]
while len(anomalies["weekend"]) < 12:
    a, d = rng.choice(staff_assignments), rng.choice(weekend_days)
    if a["assigned_from"] <= d <= a["assigned_to"]:
        r = ts_row(emp_by_id[a["employee_id"]], a["engagement_code"], d, rng.choice([3.0, 4.0, 5.5, 6.0]), "Y", "Approved", "Weekend working to meet deadline")
        timesheet_rows.append(r)
        anomalies["weekend"].append(r["timesheet_id"])

staff = [e for e in employees if e["grade_level"] <= 4]
while len(anomalies["not_assigned"]) < 10:
    emp, eng = rng.choice(staff), rng.choice(client_engagements)
    assigned = {a["engagement_code"] for a in assignments_by_emp.get(emp["employee_id"], [])}
    days = list(working_days(max(eng["start_date"], PERIOD_START), min(eng["end_date"], PERIOD_END)))
    if eng["engagement_code"] in assigned or not days:
        continue
    r = ts_row(emp, eng["engagement_code"], rng.choice(days), rng.choice([2.0, 3.0, 4.0]), "Y", "Approved", "")
    timesheet_rows.append(r)
    anomalies["not_assigned"].append(r["timesheet_id"])

# Remove month files left by an earlier run (for example the November file), so a re-run resets the folder to August-October
for old_file in glob.glob(f"{VOLUME_PATH}/timesheets/timesheets_*.csv"):
    os.remove(old_file)
write_timesheet_files(timesheet_rows)

# ---------------- Expenses (JSON lines, one file per month) ----------------
EXPENSE_CATEGORIES = [  # category, weight, amount range, merchants
    ("Rail travel", 0.33, (25, 180), ["Avanti West Coast", "LNER", "GWR", "CrossCountry", "ScotRail"]),
    ("Hotel",       0.20, (95, 210), ["Premier Inn", "Hilton", "Travelodge", "Marriott", "Holiday Inn"]),
    ("Subsistence", 0.27, (8, 45),   ["Pret A Manger", "Leon", "Wagamama", "Costa Coffee", "Client canteen"]),
    ("Taxi",        0.14, (9, 60),   ["Uber", "Addison Lee", "Local taxi"]),
    ("Mileage",     0.06, (20, 180), ["Private car"]),
]
expenses = []
for a in staff_assignments:
    eng = eng_by_code[a["engagement_code"]]
    currency = client_by_id[eng["client_id"]]["billing_currency"]
    policy = sow_terms[eng["engagement_code"]]["expense_policy"]
    week = max(a["assigned_from"], PERIOD_START)
    week -= timedelta(days=week.weekday())
    while week <= min(a["assigned_to"], PERIOD_END):
        if rng.random() < 0.35:
            d = week + timedelta(days=rng.randint(0, 4))
            cat, _, (lo, hi), merchants = rng.choices(EXPENSE_CATEGORIES, weights=[c[1] for c in EXPENSE_CATEGORIES])[0]
            if PERIOD_START <= d <= PERIOD_END:
                if cat == "Mileage":
                    miles = rng.randint(lo, hi)
                    amount, cur, desc = round(miles * 0.45, 2), "GBP", f"{miles} miles at 0.45 per mile"
                else:
                    amount, cur, desc = round(rng.uniform(lo, hi), 2), currency, f"{cat} - {eng['engagement_name']}"
                expenses.append(dict(
                    expense_id=f"EXP{len(expenses) + 1:06d}", employee_id=a["employee_id"], engagement_code=a["engagement_code"],
                    expense_date=d.isoformat(), category=cat, merchant=rng.choice(merchants), description=desc,
                    amount=amount, currency=cur, rechargeable=policy != "Expenses included in fees",
                    receipt_attached=rng.random() < 0.94, status="Approved" if rng.random() < 0.9 else "Pending",
                    submitted_at=f"{(d + timedelta(days=rng.randint(1, 10))).isoformat()}T{rng.randint(8, 19):02d}:{rng.randint(0, 59):02d}:00Z"))
        week += timedelta(days=7)

for month in sorted({e["expense_date"][:7] for e in expenses}):
    path = f"{VOLUME_PATH}/expenses/expenses_{month.replace('-', '_')}.json"
    month_rows = [e for e in expenses if e["expense_date"].startswith(month)]
    with open(path, "w") as f:
        f.writelines(json.dumps(e) + "\n" for e in month_rows)
    print(f"{path}  ({len(month_rows)} rows)")

# ---------------- Reference CSVs ----------------
with open(f"{VOLUME_PATH}/reference/grade_rate_card.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["grade", "grade_level", "standard_day_rate_gbp", "cost_day_rate_gbp", "effective_from"])
    w.writerows([g[0], g[1], g[2], g[3], "2026-04-01"] for g in GRADES)

FX = {"GBP": [1.0, 1.0, 1.0], "EUR": [0.858, 0.861, 0.866], "USD": [0.742, 0.751, 0.763]}
with open(f"{VOLUME_PATH}/reference/fx_rates_monthly.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["month", "currency", "rate_to_gbp"])
    for i, month in enumerate(["2026-08", "2026-09", "2026-10"]):
        w.writerows([month, cur, rates[i]] for cur, rates in FX.items())
print(f"{VOLUME_PATH}/reference/grade_rate_card.csv and fx_rates_monthly.csv written")
print("Seeded timesheet anomalies:", {k: len(v) for k, v in anomalies.items()})

# Checkpoint for the next cells
RNG_STATE["after_files"] = rng.getstate()
RNG_STATE["ts_counter_after_files"] = _ts_counter[0]

In [0]:
from fpdf import FPDF
from fpdf.enums import XPos, YPos

# Start from the checkpoint saved by the cell above, so re-running this cell gives the same PDFs every time
rng.setstate(RNG_STATE["after_files"])

FEE_LABEL = {"Time & Materials": "Estimated fees", "Capped T&M": "Fee cap", "Fixed Fee": "Fixed fee"}
BILLING_TEXT = {
    "Time & Materials": "Fees are charged on a time and materials basis at the agreed day rates below. The estimate is not a cap; we will notify the Client in writing if we expect it to be exceeded.",
    "Capped T&M": "Fees are charged on a time and materials basis at the agreed day rates below, up to the fee cap. Work above the cap requires a signed change request.",
    "Fixed Fee": "The fixed fee covers the scope described in this SOW. The agreed day rates below apply only to work outside scope, which needs an approved change request.",
}
SCOPE_TEXT = {
    "Audit": "Perform audit procedures in accordance with ISAs (UK), including planning, risk assessment, controls testing, substantive procedures and reporting to the Audit Committee.",
    "Risk Consulting": "Assess the design and operating effectiveness of key controls, carry out stakeholder interviews and data analysis, and present findings with prioritised recommendations.",
    "Technology Consulting": "Run discovery workshops, document current- and target-state architecture, provide independent assurance over delivery milestones and give a final recommendations report.",
    "Deal Advisory": "Review the data room, analyse quality of earnings, net debt and working capital, meet management and issue a due diligence report to the Client.",
    "Tax": "Collect supporting data, prepare computations and technical analysis, review them with the Client, and prepare submissions and documentation for HMRC.",
}
SIGNATORY_TITLES = ["Chief Financial Officer", "Group Financial Controller", "Chief Operating Officer", "Head of Internal Audit", "Chief Information Officer", "Director of Finance"]
SIGNATORY_NAMES = ["Helen Marsh", "David Okafor", "Sarah Lindqvist", "Michael Byrne", "Rachel Osei", "Andrew Kerr", "Laura Brennan", "Peter Novak"]

def fmt_date(d):
    return f"{d.day} {d.strftime('%B %Y')}"

from datetime import datetime, timezone

class SowPdf(FPDF):
    doc_ref = ""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # A fixed creation date, so regenerating gives byte-identical PDFs
        self.set_creation_date(datetime(2026, 6, 1, tzinfo=timezone.utc))
    def header(self):
        self.set_font("Helvetica", "B", 11)
        self.set_text_color(0, 51, 141)
        self.cell(0, 6, FIRM_NAME, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.set_font("Helvetica", "", 8)
        self.set_text_color(100, 100, 100)
        self.cell(0, 4, "1 Example Square, London EC2A 1AA  |  northgate-advisory.example", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.set_text_color(0, 0, 0)
        self.ln(4)
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Confidential  |  {self.doc_ref}  |  Page {self.page_no()} of {{nb}}", align="C")

def heading(pdf, text):
    pdf.ln(3)
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(0, 7, text, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("Helvetica", "", 9.5)

def kv(pdf, label, value):
    pdf.set_font("Helvetica", "B", 9.5)
    pdf.cell(55, 6, label)
    pdf.set_font("Helvetica", "", 9.5)
    pdf.cell(0, 6, str(value), new_x=XPos.LMARGIN, new_y=YPos.NEXT)

def para(pdf, text):
    pdf.multi_cell(0, 5, text, new_x=XPos.LMARGIN, new_y=YPos.NEXT)

def signatures(pdf, left, right):
    if pdf.get_y() > pdf.h - 60:  # keep both signature blocks together on one page
        pdf.add_page()
    pdf.ln(6)
    pdf.set_font("Helvetica", "", 9.5)
    y = pdf.get_y()
    for x, (org, name, title) in [(10, left), (110, right)]:
        pdf.set_xy(x, y)
        pdf.multi_cell(90, 5, f"Signed for and on behalf of {org}\n\n______________________________\n{name}\n{title}")

def build_sow_pdf(eng):
    t, client = sow_terms[eng["engagement_code"]], client_by_id[eng["client_id"]]
    partner, manager = emp_by_id[eng["engagement_partner_id"]], emp_by_id[eng["engagement_manager_id"]]
    pdf = SowPdf()
    pdf.doc_ref = eng["sow_reference"]
    pdf.set_auto_page_break(True, margin=20)
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, "STATEMENT OF WORK", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    kv(pdf, "SOW reference:", eng["sow_reference"])
    kv(pdf, "Issue date:", fmt_date(eng["start_date"] - timedelta(days=14)))
    kv(pdf, "Master Services Agreement:", f"MSA-{client['client_id']}-{client['client_since'].year}")
    kv(pdf, "Engagement code:", eng["engagement_code"])
    kv(pdf, "Engagement name:", eng["engagement_name"])

    heading(pdf, "1. Parties")
    para(pdf, f"This Statement of Work is entered into between {client['client_name']} of {client['city']}, {client['country']} "
              f"(the \"Client\") and {FIRM_NAME} (\"Northgate\"), under the Master Services Agreement referenced above.")
    heading(pdf, "2. Scope of services")
    para(pdf, f"{eng['practice']} services: {SCOPE_TEXT[eng['practice']]}")
    heading(pdf, "3. Term")
    para(pdf, f"Services will start on {fmt_date(eng['start_date'])} and are expected to finish by {fmt_date(eng['end_date'])}.")
    heading(pdf, "4. Commercial terms")
    kv(pdf, "Billing basis:", eng["billing_type"])
    kv(pdf, f"{FEE_LABEL[eng['billing_type']]}:", f"GBP {t['fee_gbp']:,} (excluding VAT)")
    kv(pdf, "Client purchase order:", t["purchase_order"])
    kv(pdf, "Payment terms:", f"{t['payment_terms_days']} days from date of invoice")
    kv(pdf, "Invoicing:", "Monthly in arrears")
    para(pdf, BILLING_TEXT[eng["billing_type"]])
    heading(pdf, "5. Agreed day rates")
    para(pdf, "One day is 7.5 hours. Time is recorded in half-hour increments. Rates are in GBP and exclude VAT.")
    pdf.ln(1)
    pdf.set_font("Helvetica", "", 9)
    with pdf.table(col_widths=(60, 45, 30, 45), text_align=("LEFT", "RIGHT", "RIGHT", "RIGHT"), line_height=6) as table:
        for vals in [("Grade", "Standard day rate (GBP)", "Discount", "Agreed day rate (GBP)")] + [
                (r["grade"], f"{r['standard_day_rate']:,}", f"{r['discount_pct']}%", f"{r['agreed_day_rate']:,}") for r in t["day_rates"]]:
            row = table.row()
            for v in vals:
                row.cell(v)
    pdf.set_font("Helvetica", "", 9.5)
    heading(pdf, "6. Expenses")
    para(pdf, f"{t['expense_policy']}. Expenses must comply with the Client travel policy and be supported by receipts.")
    heading(pdf, "7. Key personnel")
    kv(pdf, "Engagement Partner:", f"{partner['full_name']} ({partner['grade']})")
    kv(pdf, "Engagement Manager:", f"{manager['full_name']} ({manager['grade']})")
    signatures(pdf, (FIRM_NAME, partner["full_name"], partner["grade"]),
               (client["client_name"], rng.choice(SIGNATORY_NAMES), rng.choice(SIGNATORY_TITLES)))
    path = f"{VOLUME_PATH}/contracts/{eng['sow_reference']}.pdf"
    pdf.output(path)
    return path

for eng in client_engagements:
    build_sow_pdf(eng)

# A change request: a different document type in the same folder, for the classification extension
cr_eng = eng_by_code["ENG-26-005"]  # Operational Resilience Review (Capped T&M)
cr_label = FEE_LABEL[cr_eng["billing_type"]].lower()
cr_increase = 45000
pdf = SowPdf()
pdf.doc_ref = "CR-26-005-01"
pdf.add_page()
pdf.set_font("Helvetica", "B", 16)
pdf.cell(0, 10, "CHANGE REQUEST No. 1", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
kv(pdf, "Change request reference:", "CR-26-005-01")
kv(pdf, "Related SOW:", cr_eng["sow_reference"])
kv(pdf, "Engagement code:", cr_eng["engagement_code"])
kv(pdf, "Date:", "12 October 2026")
heading(pdf, "Reason for change")
para(pdf, "The Client has asked Northgate to add two more important business services (wholesale billing and network telemetry) to the scope of the resilience review, including more scenario testing.")
heading(pdf, "Commercial impact")
kv(pdf, f"Original {cr_label}:", f"GBP {sow_terms['ENG-26-005']['fee_gbp']:,}")
kv(pdf, "Increase:", f"GBP {cr_increase:,}")
kv(pdf, f"Revised {cr_label}:", f"GBP {sow_terms['ENG-26-005']['fee_gbp'] + cr_increase:,}")
para(pdf, "Agreed day rates are unchanged from the original Statement of Work.")
cr_lead = emp_by_id[cr_eng["engagement_partner_id"]]
signatures(pdf, (FIRM_NAME, cr_lead["full_name"], cr_lead["grade"]),
           (client_by_id[cr_eng["client_id"]]["client_name"], "Michael Byrne", "Chief Operating Officer"))
pdf.output(f"{VOLUME_PATH}/contracts/CR-26-005-01.pdf")

# Answer key for the AI extraction accuracy check in the tutorial notebook
with open(f"{VOLUME_PATH}/_facilitator/sow_terms_answer_key.json", "w") as f:
    f.writelines(json.dumps(dict(engagement_code=code, fee_gbp=float(t["fee_gbp"]), purchase_order=t["purchase_order"],
                                 payment_terms_days=t["payment_terms_days"])) + "\n" for code, t in sow_terms.items())
with open(f"{VOLUME_PATH}/_facilitator/sow_rates_answer_key.json", "w") as f:
    f.writelines(json.dumps(dict(engagement_code=code, grade=r["grade"], agreed_day_rate_gbp=float(r["agreed_day_rate"]))) + "\n"
                 for code, t in sow_terms.items() for r in t["day_rates"])

print(f"Wrote {len(client_engagements)} SOWs + 1 change request to {VOLUME_PATH}/contracts")
print(f"Wrote the extraction answer key to {VOLUME_PATH}/_facilitator")

# Checkpoint for the November file
RNG_STATE["after_pdfs"] = rng.getstate()

In [0]:
import shutil

# The pre-built visual data prep for the PDF extension demo (3. Extension - PDF Parsing). It's built by hand in
# Lakeflow Designer, not generated. After you change it, save a copy here so it ships with the data.
# It's stored with a .json extension so the workspace keeps it as a plain file, not a notebook.
DEMO_PREP_SOURCE = "/Volumes/praju_stable_2_catalog/consulting_billing/raw_files/_facilitator/sow_extraction_demo.designer.ipynb"
os.makedirs(f"{DATA_DIR}/designer_demo", exist_ok=True)
if os.path.exists(DEMO_PREP_SOURCE):
    shutil.copyfile(DEMO_PREP_SOURCE, f"{DATA_DIR}/designer_demo/sow_extraction_demo.designer.ipynb.json")
    print(f"Copied the Designer demo prep from {DEMO_PREP_SOURCE}")
else:
    print(f"Not found: {DEMO_PREP_SOURCE}. Keeping the copy already in data/designer_demo/")

In [0]:
import hashlib

# The November file for the scheduling exercise goes in new_month/, not timesheets/
rng.setstate(RNG_STATE["after_pdfs"])
_ts_counter[0] = RNG_STATE["ts_counter_after_files"]
for old_file in glob.glob(f"{DATA_DIR}/new_month/*.csv"):
    os.remove(old_file)
write_timesheet_files(generate_timesheet_rows(date(2026, 11, 2), date(2026, 11, 6)), f"{DATA_DIR}/new_month")

# MANIFEST.json: a checksum for every file plus the table schemas. The setup notebook uses it to load the
# tables and to prove every copy of the workshop data is byte-identical.
files = {}
for root, _, names in os.walk(DATA_DIR):
    for n in sorted(names):
        rel = os.path.relpath(os.path.join(root, n), DATA_DIR)
        if rel != "MANIFEST.json":
            files[rel] = hashlib.md5(open(os.path.join(root, n), "rb").read()).hexdigest()
with open(f"{DATA_DIR}/MANIFEST.json", "w") as f:
    json.dump(dict(files=dict(sorted(files.items())), tables=TABLE_SPECS), f, indent=2)

print(f"{len(files)} files and {len(TABLE_SPECS)} tables written to {DATA_DIR}")
for rel in sorted(files):
    print(f"  {rel:<45} {os.path.getsize(os.path.join(DATA_DIR, rel)) / 1024:>7.1f} KB")